# Notebook Description

This notebook demonstrates a full machine learning workflow for binary classification using the IoT Network Intrusion Dataset.

Steps include:
 1. Importing necessary libraries and modules for data manipulation, preprocessing, modeling, and evaluation.
 2. Loading and cleaning the dataset, handling missing and infinite values.
 3. Separating features (X) and target (y), and identifying categorical and numerical columns.
 4. Building a preprocessing pipeline using ColumnTransformer for scaling and encoding.
 5. Splitting the data into training and testing sets.
 6. Creating a pipeline that includes preprocessing, SMOTE for balancing, feature selection, and XGBoost classifier.
 7. Training the model and evaluating its performance using classification metrics, ROC-AUC, and visualizations.
 8. Performing cross-validation to assess model generalization.

# Import Libraries

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline          
import warnings; warnings.filterwarnings("ignore")


In [ ]:
import sklearn, xgboost, sys

from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier

pipe = Pipeline([
    # ... your transformers ...
    ('clf', XGBClassifier())  # <-- CORRECT: this is an instance
])

print("✅ scikit‑learn:", sklearn.__version__)
print("✅ xgboost     :", xgboost.__version__)
print("✅ python      :", sys.version)



In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd
import sklearn, xgboost


class InfToNan(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return pd.DataFrame(X).replace([np.inf, -np.inf], np.nan).values
    

print(sklearn.__version__)  # 1.4.x
print(xgboost.__version__)  # ≥ 1.7.6


# Loading and cleaning the Dataset

In [ ]:
# url to donwnload the dataset [https://www.kaggle.com/datasets/rohulaminlabid/iotid20-dataset?select=IoT+Network+Intrusion+Dataset.csv]
df = pd.read_csv("../data/IoT-Network-Intrusion-Dataset.csv").dropna()

# Separating features (X) and target (y)

In [ ]:
y = (df['Label'] == 'Anomaly').astype(int)        
X = df.drop(columns=['Label'])

X = X.replace([np.inf, -np.inf], np.nan)           
X = X.dropna()                                     
y = y[X.index]                                     


# Define columns

In [ ]:
cat_cols = X.select_dtypes('object').columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# Building a preprocessing pipeline using ColumnTransformer

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

# Splitting the data into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)


# Set up Pipeline with SMOTE then SelectKBest then XGBoost

In [ ]:
pipe = ImbPipeline(steps=[
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('kbest', SelectKBest(f_classif, k=30)),          
    ('clf',  XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42
           ))
])

# Training

In [ ]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print("Model trained successfully.")

# Evaluation on the real test

In [ ]:
# y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC‑AUC:", roc_auc_score(y_test, y_prob))

# Graphs

In [ ]:
fpr,tpr,_ = roc_curve(y_test,y_prob)
plt.plot(fpr,tpr); plt.plot([0,1],[0,1],'--'); plt.title("ROC"); plt.show()
ConfusionMatrixDisplay.from_predictions(y_test,y_pred,cmap='Blues'); plt.show()

# Cross‑validation (to prove generalization)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
print("CV F1 mean/std:", scores.mean(), scores.std())